# EASE Grid Explorer — Define Your Reconstruction Region

Use the interactive sliders to define the EASE grid region for your Arctic reconstruction.  
The shaded blue rectangle shows the extent of the grid on a polar map.

Once you're happy with the region, the last cell generates the `grid:` YAML block you can
paste into your config file.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pyproj
import ipywidgets as widgets
from IPython.display import display, clear_output

In [2]:
# ── EASE projection definition ──────────────────────────────────────────
EASE_PROJ4 = "+proj=laea +lat_0=90 +lon_0=0 +x_0=0 +y_0=0 +datum=WGS84 +units=m"
ease_crs = ccrs.LambertAzimuthalEqualArea(
    central_longitude=0, central_latitude=90,
    false_easting=0, false_northing=0,
)
proj_ease = pyproj.CRS.from_proj4(EASE_PROJ4)
proj_wgs  = pyproj.CRS.from_epsg(4326)
to_wgs    = pyproj.Transformer.from_crs(proj_ease, proj_wgs, always_xy=True)

In [3]:
# (grid_boundary_lonlat helper removed — plotting now uses native EASE coordinates)

In [4]:
# ── Plotting function ────────────────────────────────────────────────────
def plot_grid(cx, cy, w_km, h_km, res_km,
              use_latlon=False, lat_min=60, lat_max=90,
              lon_min=-180, lon_max=180):
    """Draw the Arctic map in EASE coordinates with the grid rectangle.

    If ``use_latlon`` is True, also overlay the lat/lon-bounded subregion
    that will be used by the pipeline for boundary masking.
    """
    w_m = w_km * 1000
    h_m = h_km * 1000

    # Axes extent: pad around the rectangle itself (asymmetric — follows the
    # rectangle when off-center instead of forcing a symmetric square).
    pad = max(w_m, h_m) * 0.3
    x_lo, x_hi = cx - w_m / 2 - pad, cx + w_m / 2 + pad
    y_lo, y_hi = cy - h_m / 2 - pad, cy + h_m / 2 + pad
    # Keep a square aspect so the polar map isn't distorted
    half = max(x_hi - x_lo, y_hi - y_lo) / 2
    xc, yc = (x_lo + x_hi) / 2, (y_lo + y_hi) / 2
    x_lo, x_hi = xc - half, xc + half
    y_lo, y_hi = yc - half, yc + half

    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(1, 1, 1, projection=ease_crs)
    ax.set_extent([x_lo, x_hi, y_lo, y_hi], crs=ease_crs)

    # Coastlines, land, ocean — cartopy handles projection automatically
    ax.add_feature(cfeature.OCEAN, facecolor='#e6f2ff')
    ax.add_feature(cfeature.LAND,  facecolor='#f0ebe0', edgecolor='0.4', linewidth=0.4)
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.5,
                 xlocs=range(-180, 180, 30), ylocs=range(50, 91, 10))

    # Grid rectangle — directly in EASE metres (native coordinates)
    rect = mpatches.Rectangle(
        (cx - w_m / 2, cy - h_m / 2), w_m, h_m,
        facecolor='royalblue', alpha=0.25,
        edgecolor='royalblue', linewidth=1.5,
        transform=ease_crs,
    )
    ax.add_patch(rect)

    # Optional lat/lon-bounded subregion overlay (orange polygon, drawn in PlateCarree
    # so cartopy correctly curves it onto the EASE projection)
    latlon_info = ''
    n_in_mask = None
    if use_latlon:
        n_edge = 200
        lons_poly = np.concatenate([
            np.linspace(lon_min, lon_max, n_edge),     # south edge
            np.full(n_edge, lon_max),                  # east edge
            np.linspace(lon_max, lon_min, n_edge),     # north edge
            np.full(n_edge, lon_min),                  # west edge
        ])
        lats_poly = np.concatenate([
            np.full(n_edge, lat_min),
            np.linspace(lat_min, lat_max, n_edge),
            np.full(n_edge, lat_max),
            np.linspace(lat_max, lat_min, n_edge),
        ])
        ax.fill(lons_poly, lats_poly,
                facecolor='darkorange', alpha=0.35,
                edgecolor='darkorange', linewidth=1.5,
                transform=ccrs.PlateCarree(), zorder=5)
        latlon_info = (f'   lat=[{lat_min:g}, {lat_max:g}]°   '
                       f'lon=[{lon_min:g}, {lon_max:g}]°')

        # Count grid cells whose centers fall inside the lat/lon bbox.
        # Vectorised through pyproj — fast even for ~80k cells.
        nx_f = w_km / res_km
        ny_f = h_km / res_km
        if nx_f == int(nx_f) and ny_f == int(ny_f):
            nx_c, ny_c = int(nx_f), int(ny_f)
            res_m = res_km * 1000
            xc_cells = cx - w_m / 2 + (np.arange(nx_c) + 0.5) * res_m
            yc_cells = cy - h_m / 2 + (np.arange(ny_c) + 0.5) * res_m
            X, Y = np.meshgrid(xc_cells, yc_cells)
            lons_c, lats_c = to_wgs.transform(X.ravel(), Y.ravel())
            in_mask = ((lats_c >= lat_min) & (lats_c <= lat_max) &
                       (lons_c >= lon_min) & (lons_c <= lon_max))
            n_in_mask = int(in_mask.sum())

    # Grid center
    ax.plot(cx, cy, 'r+', markersize=12, markeredgewidth=2, transform=ease_crs)

    # Info text
    nx = w_km / res_km
    ny = h_km / res_km
    ok = (nx == int(nx) and ny == int(ny))
    nx_i, ny_i = int(nx), int(ny)
    total = nx_i * ny_i
    if ok:
        if n_in_mask is not None:
            pct = 100.0 * n_in_mask / total if total else 0.0
            status = (f'{nx_i} × {ny_i} = {total} cells   '
                      f'(mask: {n_in_mask} cells, {pct:.1f}%)')
        else:
            status = f'{nx_i} × {ny_i} cells'
    else:
        status = f'{nx:.1f} × {ny:.1f} — NOT divisible!'
    color = 'black' if ok else 'red'

    clon, clat = to_wgs.transform(cx, cy)
    info = (f'Center: ({cx/1000:.0f}, {cy/1000:.0f}) km   '
            f'Extent: {w_km} × {h_km} km\n'
            f'Resolution: {res_km} km   →   {status}{latlon_info}\n'
            f'Center lat/lon: {clat:.2f}°N, {clon:.2f}°E')
    ax.set_title(info, fontsize=10, color=color, loc='left', pad=10)

    plt.tight_layout()
    plt.show()


## (Optional) Initialize from an existing config

Paste a previously-generated `grid:` YAML block (or the `grid:` section of an existing
config file) into the cell below to pre-fill the sliders. Leave the default to start from
scratch. Then run the **widget cell** that follows to see the corresponding display.


In [16]:
# ── Optional: paste a `grid:` YAML block here to pre-fill the sliders ────
# Leave config_text empty (or just whitespace) to use the built-in defaults.

config_text = """
# Grid: 200 × 192 = 38400 cells
# Lat/lon mask: 20493 cells (53.4% of grid)
grid:
  resolution_km: 3.125
  center_x_m: -2150000
  center_y_m: -1400000
  width_km: 625       # 200 cells × 3.125 km
  height_km: 600      # 192 cells × 3.125 km
  lat_min: 65
  lat_max: 68
  lon_min: -64
  lon_max: -50
"""

import re

def _parse_grid_block(text):
    """Lightweight YAML parser for the flat `grid:` block we generate."""
    params = {}
    in_grid = False
    for raw in text.splitlines():
        line = raw.split('#', 1)[0].rstrip()      # strip trailing comments
        if not line.strip():
            continue
        if re.match(r'^\s*grid\s*:\s*$', line):
            in_grid = True
            continue
        if in_grid and not line.startswith((' ', '\t')):
            break                                  # left the grid: block
        m = re.match(r'^\s+([A-Za-z_]\w*)\s*:\s*(-?[\d.]+)\s*$', line)
        if m:
            key, val = m.group(1), m.group(2)
            params[key] = float(val) if '.' in val else int(val)
    return params

init_params = _parse_grid_block(config_text)
print("Loaded init_params:", init_params if init_params else "(none — using defaults)")


Loaded init_params: {'resolution_km': 3.125, 'center_x_m': -2150000, 'center_y_m': -1400000, 'width_km': 625, 'height_km': 600, 'lat_min': 65, 'lat_max': 68, 'lon_min': -64, 'lon_max': -50}


In [17]:
# ── Interactive widget ────────────────────────────────────────────────────
style  = {'description_width': '140px'}
layout = widgets.Layout(width='500px')

# Pull initial values from a pasted config (if any), else use defaults.
_p = globals().get('init_params', {}) or {}

# Snap requested resolution to the nearest allowed option
_res_opts = [1, 3, 3.125, 6.25, 9, 12.5, 25]
_res0 = _p.get('resolution_km', 25)
_res0 = min(_res_opts, key=lambda o: abs(o - _res0))

_ll_keys = ('lat_min', 'lat_max', 'lon_min', 'lon_max')
_use_ll0 = any(k in _p for k in _ll_keys)

s_res = widgets.SelectionSlider(
    options=_res_opts,
    value=_res0, description='Resolution (km)', style=style, layout=layout,
    continuous_update=False)

s_cx = widgets.IntSlider(
    value=int(_p.get('center_x_m', 0) / 1000), min=-4000, max=4000, step=50,
    description='Center X (km)', style=style, layout=layout,
    continuous_update=False)

s_cy = widgets.IntSlider(
    value=int(_p.get('center_y_m', 0) / 1000), min=-4000, max=4000, step=50,
    description='Center Y (km)', style=style, layout=layout,
    continuous_update=False)

s_w = widgets.IntSlider(
    value=int(_p.get('width_km', 8750)), min=100, max=12000, step=25,
    description='Width (km)', style=style, layout=layout,
    continuous_update=False)

s_h = widgets.IntSlider(
    value=int(_p.get('height_km', 8750)), min=100, max=12000, step=25,
    description='Height (km)', style=style, layout=layout,
    continuous_update=False)

# ── Lat/lon boundary-masking controls ────────────────────────────────────
s_use_ll = widgets.Checkbox(
    value=_use_ll0, description='Apply lat/lon mask',
    style=style, layout=layout)

s_lat_min = widgets.FloatSlider(
    value=float(_p.get('lat_min', 60.0)), min=-90.0, max=90.0, step=0.5,
    description='lat_min (°N)', style=style, layout=layout,
    continuous_update=False, readout_format='.1f')

s_lat_max = widgets.FloatSlider(
    value=float(_p.get('lat_max', 90.0)), min=-90.0, max=90.0, step=0.5,
    description='lat_max (°N)', style=style, layout=layout,
    continuous_update=False, readout_format='.1f')

s_lon_min = widgets.FloatSlider(
    value=float(_p.get('lon_min', -180.0)), min=-180.0, max=180.0, step=1.0,
    description='lon_min (°E)', style=style, layout=layout,
    continuous_update=False, readout_format='.1f')

s_lon_max = widgets.FloatSlider(
    value=float(_p.get('lon_max', 180.0)), min=-180.0, max=180.0, step=1.0,
    description='lon_max (°E)', style=style, layout=layout,
    continuous_update=False, readout_format='.1f')

out = widgets.Output()

def on_change(_):
    with out:
        clear_output(wait=True)
        plot_grid(
            cx=s_cx.value * 1000,
            cy=s_cy.value * 1000,
            w_km=s_w.value,
            h_km=s_h.value,
            res_km=s_res.value,
            use_latlon=s_use_ll.value,
            lat_min=s_lat_min.value,
            lat_max=s_lat_max.value,
            lon_min=s_lon_min.value,
            lon_max=s_lon_max.value,
        )

for s in [s_res, s_cx, s_cy, s_w, s_h,
          s_use_ll, s_lat_min, s_lat_max, s_lon_min, s_lon_max]:
    s.observe(on_change, names='value')

# Initial plot
on_change(None)

ll_box = widgets.VBox(
    [s_use_ll, s_lat_min, s_lat_max, s_lon_min, s_lon_max],
    layout=widgets.Layout(border='1px solid lightgray',
                          padding='5px', margin='5px 0'))

display(widgets.VBox([s_res, s_cx, s_cy, s_w, s_h, ll_box, out]))


## Export grid parameters

Run the cell below to print the YAML `grid:` block matching the current slider values.  
Copy-paste it into your config file.

In [18]:
cx_m = s_cx.value * 1000
cy_m = s_cy.value * 1000
res  = s_res.value
w    = s_w.value
h    = s_h.value
nx   = int(w / res)
ny   = int(h / res)
total = nx * ny

lines = [
    f"# Grid: {nx} × {ny} = {total} cells",
]

# Mask cell count (centers inside lat/lon bbox)
if s_use_ll.value:
    res_m = res * 1000
    w_m, h_m = w * 1000, h * 1000
    xc_cells = cx_m - w_m / 2 + (np.arange(nx) + 0.5) * res_m
    yc_cells = cy_m - h_m / 2 + (np.arange(ny) + 0.5) * res_m
    X, Y = np.meshgrid(xc_cells, yc_cells)
    lons_c, lats_c = to_wgs.transform(X.ravel(), Y.ravel())
    in_mask = ((lats_c >= s_lat_min.value) & (lats_c <= s_lat_max.value) &
               (lons_c >= s_lon_min.value) & (lons_c <= s_lon_max.value))
    n_in = int(in_mask.sum())
    pct = 100.0 * n_in / total if total else 0.0
    lines.append(f"# Lat/lon mask: {n_in} cells ({pct:.1f}% of grid)")

lines += [
    "grid:",
    f"  resolution_km: {res}",
    f"  center_x_m: {cx_m}",
    f"  center_y_m: {cy_m}",
    f"  width_km: {w}       # {nx} cells × {res} km",
    f"  height_km: {h}      # {ny} cells × {res} km",
]
if s_use_ll.value:
    lines += [
        f"  lat_min: {s_lat_min.value:g}",
        f"  lat_max: {s_lat_max.value:g}",
        f"  lon_min: {s_lon_min.value:g}",
        f"  lon_max: {s_lon_max.value:g}",
    ]
print("\n".join(lines))


# Grid: 96 × 96 = 9216 cells
# Lat/lon mask: 5119 cells (55.5% of grid)
grid:
  resolution_km: 6.25
  center_x_m: -2200000
  center_y_m: -1400000
  width_km: 600       # 96 cells × 6.25 km
  height_km: 600      # 96 cells × 6.25 km
  lat_min: 65
  lat_max: 68
  lon_min: -64
  lon_max: -50
